# 2 · Autoformalisation by precedent

The loop:

```
provision text
    ├─ triage      does this allocate a loss?  which factors?
    ├─ retrieve    binding precedents and near misses, by constraint distance
    ├─ formalise   candidate Holding as JSON
    ├─ verify      Lean: compiles?  well-formed?  coherent with the corpus?
    └─ repair      hand the model its own errors, up to N times
```

The verification step is what makes this different from schema-constrained
generation. Three checks run, and only the first is what autoformalisation work
usually means by "verified":

1. **It compiles.** The factors exist, the remedy is a remedy.
2. **It is well-formed as a holding.** Every factor relied on must be present
   *and favour the party who won*. This catches reasoning errors, not syntax
   errors — a model that reads "the animal was innocuous, so he pays only half"
   and records `behavedAnomalously` as a reason for the *claimant* fails here.
   (It caught exactly that mistake in our own hand-formalisation of m.BK 1:1.)
3. **It coheres with the tradition.** Adding it must not make the tradition's
   case base force both outcomes on some situation it already addresses.

Requires the Lean toolchain. `elan`, then `lake build` in `lean/` -- about
seven seconds, because there is no Mathlib dependency.

In [1]:
import os, sys, json, subprocess
from pathlib import Path

# Locate the repository root whether this runs from notebooks/ or the root.
ROOT = Path.cwd()
while not (ROOT / "src" / "nomos").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
os.chdir(ROOT)
print("repo root:", ROOT)

# The built corpus is a reproducible artefact and is not shipped in the
# archive (it is 13 MB gzipped and rebuilds from data/raw in about a minute).
if not any((ROOT / "data/corpus").glob("provisions.jsonl*")):
    print("building the corpus from data/raw ...")
    subprocess.run([sys.executable, "-m", "nomos.build_corpus"], check=True,
                   env={**os.environ, "PYTHONPATH": str(ROOT / "src")})


repo root: /home/claude/nomos


In [2]:
from nomos.verify import lean_available, build_library, export_seed
print("lean available:", lean_available())
if lean_available():
    ok, log = build_library()
    print("library builds:", ok)
    if not ok:
        print(log[-2000:])

lean available: True


library builds: True


In [3]:
from nomos.schema import read_jsonl
from nomos.analogy import Holding

seed_rows = list(read_jsonl("data/seed/formalizations.jsonl"))
seed = [Holding(cite=r["cite"], situation=r["situation"], winner=r["winner"],
                remedy=r["remedy"], reason=r["reason"], tradition=r["tradition"],
                restatement=(r.get("text") or "")[:300])
        for r in seed_rows]
texts = {r["cite"]: (r.get("text") or "") for r in seed_rows}
print(len(seed), "seed holdings")

60 seed holdings


## The verifier, on three candidates

A good one, a reasoning error, and a syntax error.

In [4]:
from nomos.verify import verify

good = Holding(
    cite="TEST: forewarned beast, properly confined, escapes in a storm",
    situation=["harmOccurred","respondentsInstrument","harmToPerson","knownVice",
               "warned","properPrecaution","irresistibleForce"],
    winner="respondent", remedy="Remedy.exempt",
    reason=["properPrecaution","irresistibleForce"], tradition="rabbinic")

bad = Holding(   # a respondent's defence used as the claimant's ratio
    cite="TEST: malformed",
    situation=["harmOccurred","respondentsInstrument","behavedAnomalously"],
    winner="claimant", remedy='(Remedy.compensate ⟨"the damage"⟩)',
    reason=["behavedAnomalously"], tradition="rabbinic")

ugly = Holding(  # a factor that does not exist
    cite="TEST: bad factor", situation=["harmOccurred","notAFactor"],
    winner="claimant", remedy="Remedy.exempt", reason=["harmOccurred"],
    tradition="roman")

for label, h in [("good", good), ("reasoning error", bad), ("syntax error", ugly)]:
    r = verify(h, h.tradition)
    print(f"{label:16s} {r.summary()}")
    for e in r.errors[:2]:
        print(f"{'':16s}   {e[:110]}")
    print()

good             compiles=True well_formed=True coheres=True novel=True



reasoning error  compiles=False well_formed=False coheres=True novel=False errors=1
                   tactic 'decide' proved that the proposition



syntax error     compiles=False well_formed=False coheres=None novel=None errors=2
                   unknown constant 'Nomos.Factor.notAFactor'
                   aborting evaluation since the expression depends on the 'sorry' axiom, which can lead to runtime instability a



Notice what the middle case shows. The candidate is *syntactically perfect* —
every name exists, the remedy is well-typed — and it still fails, because
`behavedAnomalously` is a reason for the respondent and cannot be a reason the
claimant won. That check is the whole argument for targeting a proof assistant
rather than a JSON schema.

## The prompt

Built from the formal apparatus rather than from similarity. Have a look at
what the model actually sees.

In [5]:
from nomos import prompts
from nomos.schema import read_jsonl

provisions = list(read_jsonl("data/corpus/provisions.jsonl"))
target = next(p for p in provisions if p["canonical"] == "D.9.2.31")   # the pruner

guess = ["harmOccurred", "respondentActedDirectly", "harmToPerson",
         "noPrecaution", "inPublicOrClaimantsGround"]
user = prompts.build_prompt(target, seed, guess_situation=guess)
print(user[user.index("PRECEDENT ALREADY"):][:2600])

PRECEDENT ALREADY FORMALISED FOR THIS FACT PATTERN

[1] XII Tab. 8.3   (roman)   -- binding
    why shown: binds: every factor of its ratio ['harmToPerson', 'respondentActedDirectly'] is present, and this case raises no counter-reason it did not face
    holds: Manu fustive si os fregit libero, CCC <assium>, si servo, CL <assium> poenam subito si iniuriam faxsit, viginti quinque poenae <asses> sunto.
    situation: ['harmOccurred', 'harmToPerson', 'respondentActedDirectly']
    winner:    claimant
    remedy:    (Remedy.tariff ⟨300, ⟨"as"⟩⟩)
    ratio:     ['harmToPerson', 'respondentActedDirectly']

[2] Ex 21:33-34   (covenant)   -- binding
    why shown: binds: every factor of its ratio ['respondentActedDirectly', 'noPrecaution'] is present, and this case raises no counter-reason it did not face
    holds: And if a man shall open a pit, or if a man shall dig a pit and not cover it, and an ox or an ass fall therein,
the owner of the pit shall make it good; he shall give money unto the

## Running the pipeline

`EchoBackend` needs no network and is what the tests use. For real runs, set an
API key and swap in `AnthropicBackend` or `OpenAIBackend`, or point
`HFBackend` at the model fine-tuned in notebook 3.

In [6]:
from nomos.pipeline import (AnthropicBackend, EchoBackend, OpenAIBackend,
                            HFBackend, run, formalize)

if os.environ.get("ANTHROPIC_API_KEY"):
    backend = AnthropicBackend(model="claude-sonnet-4-5")
    print("using Anthropic")
elif os.environ.get("OPENAI_API_KEY"):
    backend = OpenAIBackend()
    print("using OpenAI")
else:
    backend = EchoBackend(answers={
        "PROVISION TO FORMALISE": json.dumps({
            "applicable": True,
            "restatement": ("A pruner who throws down a branch without calling out "
                            "is liable where the branch falls in a public place, and "
                            "on private ground too if a careful man would have "
                            "foreseen the danger."),
            "situation": ["harmOccurred", "respondentActedDirectly", "harmToPerson",
                          "noPrecaution", "inPublicOrClaimantsGround"],
            "winner": "claimant",
            "remedy": '(Remedy.compensate ⟨"the damage"⟩)',
            "reason": ["respondentActedDirectly", "noPrecaution"],
            "confidence": "high",
            "notes": "Mucius extends the rule to private ground by a foreseeability test."
        }),
    }, default=json.dumps({"allocates_loss": True,
                           "factors": ["harmOccurred", "respondentActedDirectly",
                                       "harmToPerson", "noPrecaution",
                                       "inPublicOrClaimantsGround"],
                           "fact_pattern": "FP-WRONGFUL-DAMAGE",
                           "one_line": "the pruner who fells a branch without warning"}))
    print("no API key found -- using EchoBackend with a scripted answer")

no API key found -- using EchoBackend with a scripted answer


In [7]:
out = formalize(target, seed, backend, max_repairs=2,
                verify_with_lean=lean_available())
print("status:    ", out.status)
print("retrieved: ", out.retrieved[:5])
if out.verification:
    print("verified:  ", out.verification.summary())
if out.holding:
    print()
    from nomos.verify import render
    print(render(out.holding))

status:     accepted
retrieved:  ['XII Tab. 8.3', 'Ex 21:33-34', 'D.9.2.27.5 (Ulpian) -- lex Aquilia c.3', 'Ex 22:5', 'D.9.2.44pr (Ulpian)']
verified:   compiles=True well_formed=True coheres=True novel=False

def candidate : Precedent :=
  { cite := "Dig. 9.2.31"
  , situation := ⟨[Factor.harmOccurred, Factor.respondentActedDirectly, Factor.harmToPerson, Factor.noPrecaution, Factor.inPublicOrClaimantsGround]⟩
  , winner := Side.claimant
  , remedy := (Remedy.compensate ⟨"the damage"⟩)
  , reason := [Factor.respondentActedDirectly, Factor.noPrecaution] }



## Evaluation

The metric that matters is stratified. A system that copies the nearest binding
precedent scores perfectly on provisions whose outcome the corpus already
forced, and learns nothing. So results are reported separately for:

* **forced** — the case base already determined the outcome. Cheap.
* **open** — nothing settled it. The model had to read the text.

`novelty_gap` is the difference. A large positive gap means the system is
retrieving rather than reading.

In [8]:
from nomos import baselines, eval as ev

base, held = ev.holdout_split(seed, fraction=0.35, seed_value=7)
print(f"case base {len(base)}  |  held out {len(held)}")
print("strata:", {"open": sum(1 for h in held if ev.is_open(base, h.situation)),
                  "forced": sum(1 for h in held if not ev.is_open(base, h.situation))})
print()

lex = baselines.make_lexical(base, texts)
rows = []
for name, fn in [("majority", baselines.majority),
                 ("nearest_precedent", baselines.nearest_precedent),
                 ("lexical_nn", lex)]:
    preds = {h.cite: fn(h.situation, base,
                        {"citation": h.cite, "text": texts.get(h.cite, "")})
             for h in held}
    rep = ev.evaluate(held, preds, base)
    o, f = rep.by_stratum.get("open", {}), rep.by_stratum.get("forced", {})
    rows.append((name, o.get("outcome", 0), f.get("outcome", 0),
                 o.get("ratio_jaccard", 0), rep.novelty_gap() or 0))

print(f"{'system':20s} {'open':>7s} {'forced':>8s} {'ratio(open)':>12s} {'gap':>7s}")
for r in rows:
    print(f"{r[0]:20s} {r[1]:7.2f} {r[2]:8.2f} {r[3]:12.2f} {r[4]:+7.2f}")

case base 39  |  held out 21
strata: {'open': 5, 'forced': 16}

system                  open   forced  ratio(open)     gap
majority                0.20     1.00         0.00   +0.80
nearest_precedent       0.40     1.00         0.30   +0.60
lexical_nn              0.40     0.81         0.10   +0.41


**Read these numbers with the sample size in mind.** The held-out set is about
fifteen holdings, so a single case moves a number by seven points. They are here
to show the harness works and that the stratification bites, not to establish
anything about any system. Getting the seed corpus to a few hundred holdings is
the first thing worth doing with this repository.

What the table does show: `nearest_precedent` gets everything right on the
forced stratum and much less on the open one. Any evaluation that pooled the
two would have reported it as a strong system.

## A full run

Formalising a slice of the Digest, growing the case base as we go. This costs
real API calls if a key is set, so it is deliberately small — raise `LIMIT` to
scale it up.

In [9]:
LIMIT = 8
candidates = [p for p in provisions
              if p["work"] == "Digesta Iustiniani"
              and p["canonical"].startswith("D.9.2.")
              and p["kind"] == "provision"
              and 120 < len(p["text"] or "") < 900][:LIMIT]
print(f"{len(candidates)} candidate provisions")
for p in candidates[:3]:
    print(f"  {p['citation']:16s} {p['text'][:90]}...")

8 candidate provisions
  Dig. 9.2.1pr     Lex aquilia omnibus legibus, quae ante se de damno iniuria locutae sunt, derogavit, sive d...
  Dig. 9.2.2pr     Lege aquilia capite primo cavetur: " ut qui servum servamve alienum alienamve quadrupedem ...
  Dig. 9.2.2.2     Ut igitur apparet, servis nostris exaequat quadrupedes, quae pecudum numero sunt et gregat...


In [10]:
# Uncomment to run.  With EchoBackend this returns 'not-applicable' for
# everything, which is the correct behaviour for a backend that knows nothing.
#
# results = run(candidates, seed, backend, max_repairs=2,
#               verify_with_lean=lean_available(),
#               on_result=lambda o: print(f"{o.citation:18s} {o.status:16s} "
#                                         f"{(o.verification.summary() if o.verification else '')}"))
# import collections; print(collections.Counter(o.status for o in results))
#
# Path("data/runs").mkdir(parents=True, exist_ok=True)
# with open("data/runs/digest_9_2.jsonl", "w") as fh:
#     for o in results:
#         fh.write(json.dumps(o.to_json(), ensure_ascii=False) + "\n")
print("ready")

ready
